# Code mẫu — Bài tập lý thuyết

Các ví dụ dưới đây nối những đại lượng tính trên giấy với code: mẫu số của xác suất có điều kiện, đơn vị của log-loss, và gradient tính đúng theo quy ước lấy trung bình. Mỗi ví dụ có thêm một cách tính thứ hai để đối chiếu. Code chạy được vẫn có thể sai, và cách tính thứ hai giúp lộ ra những lỗi như vậy.

## Ví dụ 1: Bắt được 80% bài lỗi không có nghĩa 80% cảnh báo là đúng

Trong 100 bài nộp có 10 bài lỗi. Bộ phát hiện cảnh báo 8 bài lỗi, và cảnh báo nhầm 18 bài không lỗi. Nếu chọn ngẫu nhiên một bài trong số bị cảnh báo, xác suất bài đó thật sự lỗi có mẫu số là $8+18$, chứ không phải 10 như khi tính tỉ lệ bắt lỗi. Ta tính bằng công thức Bayes, rồi đối chiếu với một bảng nhãn cụ thể.

In [ ]:
import numpy as np
from sklearn.metrics import precision_score, recall_score

tp, fn, fp, tn = 8, 2, 18, 72
prior = (tp + fn) / (tp + fn + fp + tn)
sensitivity = tp / (tp + fn)
false_alarm = fp / (fp + tn)
alarm = prior * sensitivity + (1 - prior) * false_alarm
posterior = prior * sensitivity / alarm
labels = np.array([1] * 10 + [0] * 90)
predicted = np.array([1] * 8 + [0] * 2 + [1] * 18 + [0] * 72)
assert np.isclose(posterior, precision_score(labels, predicted))
assert np.isclose(sensitivity, recall_score(labels, predicted))
print("P(cảnh báo | lỗi):", f"{sensitivity:.4f}")
print("P(lỗi | cảnh báo):", f"{posterior:.4f}")
print("kiểm bằng đếm:", tp, "/", tp + fp)

# Chỉ đổi tỉ lệ bài lỗi. Khả năng bắt lỗi và tỉ lệ báo nhầm giữ nguyên.
new_prior = 0.5
new_posterior = new_prior * sensitivity / (
    new_prior * sensitivity + (1 - new_prior) * false_alarm)
print("nếu P(lỗi)=0.5, posterior:", f"{new_posterior:.4f}")

P(cảnh báo | lỗi): 0.8000
P(lỗi | cảnh báo): 0.3077
kiểm bằng đếm: 8 / 26
nếu P(lỗi)=0.5, posterior: 0.8000


Trong các bài lỗi, 80% bị cảnh báo. Đó là recall. Nhưng trong các bài bị cảnh báo, chỉ $8/26\approx0.31$ là lỗi thật. Đó là precision. Hai con số trả lời hai câu hỏi khác nhau, vì mẫu số khác nhau. Công thức Bayes và cách đếm trực tiếp cho cùng kết quả, và cả hai khớp với scikit-learn.

Phần cuối đổi tỉ lệ bài lỗi từ 10% lên 50%, giữ nguyên bộ phát hiện. Lúc này, 80% cảnh báo là đúng. Cùng một bộ phát hiện, ý nghĩa của một cảnh báo thay đổi hẳn chỉ vì tỉ lệ lỗi ban đầu khác đi.

## Ví dụ 2: Nhãn giống nhau, mức tự tin khác nhau

Bốn nhãn thật là `0, 1, 1, 0`. Model A cho xác suất lớp 1 là `0.1, 0.9, 0.4, 0.6`. Model B cho `0.01, 0.99, 0.01, 0.99`. Với ngưỡng 0.5, hai model đúng ở cùng hai mẫu và sai ở cùng hai mẫu. Nhưng B sai một cách cực kỳ tự tin. Log-loss, tính bằng log tự nhiên nên có đơn vị là nats, sẽ thấy điều mà accuracy bỏ qua.

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, log_loss

y = np.array([0, 1, 1, 0])
models = {"A": np.array([0.1, 0.9, 0.4, 0.6]),
          "B": np.array([0.01, 0.99, 0.01, 0.99])}
for name, p in models.items():
    per_row = -y * np.log(p) - (1 - y) * np.log1p(-p)
    library_loss = log_loss(y, np.column_stack([1 - p, p]), labels=[0, 1])
    assert np.isclose(per_row.mean(), library_loss)
    print(name, "loss từng mẫu:", per_row.round(4).tolist())
    print("accuracy:", f"{accuracy_score(y, p >= 0.5):.2f}",
          "mean loss (nats):", f"{library_loss:.4f}")
assert np.array_equal(models["A"] >= 0.5, models["B"] >= 0.5)

q = 0.25
expected = lambda p: -q * np.log(p) - (1 - q) * np.log1p(-p)
print("dự đoán cố định, loss kỳ vọng tại p=q / p=0.5:",
      f"{expected(q):.4f}", f"{expected(0.5):.4f}")
assert expected(q) < expected(0.5)

A loss từng mẫu: [0.1054, 0.1054, 0.9163, 0.9163]
accuracy: 0.50 mean loss (nats): 0.5108
B loss từng mẫu: [0.0101, 0.0101, 4.6052, 4.6052]
accuracy: 0.50 mean loss (nats): 2.3076
dự đoán cố định, loss kỳ vọng tại p=q / p=0.5: 0.5623 0.6931


Hai model có cùng accuracy 0.5. Ở hai mẫu đúng, B được thưởng hơn một chút vì tự tin hơn. Nhưng ở hai mẫu sai, B bị phạt khoảng 4.6 nats mỗi mẫu, so với 0.92 của A. Tính trung bình, log-loss của B gấp hơn bốn lần A. Log-loss giữ lại thông tin về độ tự tin, thứ đã mất khi đổi xác suất thành nhãn cứng.

Phần cuối xét một tình huống khác: chưa biết nhãn của mẫu nào, chỉ biết 25% mẫu là lớp 1. Nếu phải đoán một xác suất cố định cho mọi mẫu, đoán đúng 0.25 cho loss kỳ vọng thấp nhất, 0.5623, thấp hơn đoán 0.5. Đây là tính chất của log-loss: nó thưởng cho việc nói đúng xác suất thật. Phép tính này là kỳ vọng trước khi có nhãn, không thay được việc chấm từng dự đoán trên nhãn thật.

## Ví dụ 3: Gradient có cả hệ số trung bình lẫn regularization

Batch có hai hàng đặc trưng `(1, 2)` và `(2, −1)`, nhãn `1, 0`. Loss là **trung bình** binary cross-entropy trên hai mẫu, cộng $0.1\cdot\sum w^2$. Bias không bị phạt. Tính tay, mỗi mẫu đóng góp $(p-y)x$ vào gradient của trọng số. Lấy trung bình hai mẫu, rồi cộng thêm $0.2w$ từ phần phạt. Ta kiểm công thức này bằng sai phân trung tâm, và bằng autograd với số thực 64 bit.

In [ ]:
import numpy as np
import torch

torch.set_num_threads(1)
torch.manual_seed(93)
X = np.array([[1.0, 2.0], [2.0, -1.0]])
y = np.array([1.0, 0.0])
theta = np.array([0.2, -0.1, 0.0])  # Hai trọng số, rồi tới bias.
penalty = 0.1

def loss(parameters):
    z = X @ parameters[:2] + parameters[2]
    return np.mean(np.logaddexp(0.0, z) - y * z) + penalty * np.sum(parameters[:2] ** 2)

z = X @ theta[:2] + theta[2]
p = 1 / (1 + np.exp(-z))
gradient = np.append(X.T @ (p - y) / len(y) + 2 * penalty * theta[:2],
                     np.mean(p - y))
epsilon = 1e-5
numerical = []
for column in range(len(theta)):
    shift = np.zeros_like(theta)
    shift[column] = epsilon
    numerical.append((loss(theta + shift) - loss(theta - shift)) / (2 * epsilon))

parameters = torch.tensor(theta, dtype=torch.float64, requires_grad=True)
features = torch.tensor(X, dtype=torch.float64)
labels = torch.tensor(y, dtype=torch.float64)
logits = features @ parameters[:2] + parameters[2]
torch_loss = torch.nn.functional.binary_cross_entropy_with_logits(logits, labels)
torch_loss = torch_loss + penalty * parameters[:2].square().sum()
torch_loss.backward()
assert np.allclose(gradient, numerical, rtol=0, atol=1e-8)
assert np.allclose(gradient, parameters.grad.numpy(), rtol=0, atol=1e-12)
print("gradient (w0,w1,b):", gradient.round(6).tolist())
print("hai phép kiểm khớp:", True)
for learning_rate in [0.1, 20.0]:
    updated = theta - learning_rate * gradient
    print("lr:", learning_rate, "loss:", f"{loss(theta):.4f}", "->", f"{loss(updated):.4f}")
assert loss(theta - 0.1 * gradient) < loss(theta)
assert loss(theta - 20.0 * gradient) > loss(theta)
large_step = theta - 20.0 * gradient
large_logits = X @ large_step[:2] + large_step[2]
data_loss = np.mean(np.logaddexp(0.0, large_logits) - y * large_logits)
l2_loss = penalty * np.sum(large_step[:2] ** 2)
print("lr=20, data loss / L2:", f"{data_loss:.6f}", f"{l2_loss:.6f}")

gradient (w0,w1,b): [0.412459, -0.83123, 0.06123]
hai phép kiểm khớp: True
lr: 0.1 loss: 0.8386 -> 0.7556
lr: 20.0 loss: 0.8386 -> 33.7852
lr=20, data loss / L2: 0.000000 33.785159


Gradient có ba thành phần, ứng với hai trọng số và bias. Bias không bị phạt, nên gradient của nó chỉ là trung bình của $p-y$. Hai cách kiểm đi theo hai con đường khác hẳn nhau. Sai phân trung tâm đo trực tiếp loss đổi bao nhiêu khi nhích từng tham số. Autograd đi ngược qua các phép tính. Cả hai đều khớp với công thức viết tay. Tham số `rtol=0` buộc so sánh chỉ theo sai số tuyệt đối đã chọn.

Bước học 0.1 làm loss giảm, đúng như mong đợi. Bước học 20 thì khác. Hai dòng cuối cho thấy phần loss trên dữ liệu gần như về 0: model giờ đoán đúng cả hai mẫu, và rất tự tin. Nhưng trọng số đã phình to, nên phần phạt L2 vọt lên khoảng 33.8, kéo tổng loss tăng mạnh. Gradient đúng chỉ cho biết hướng đi ở gần điểm hiện tại. Nó không bảo đảm một bước dài tùy ý vẫn làm loss giảm.